In [2]:
import pandas as pd
import numpy as np
import requests
import os
import json

## Flightaware AeroAPI

In [3]:
with open('../apiKeys/apiKeys.json') as f:
    API_KEYS = json.load(f)
    f.close()

In [7]:
AEROAPI_BASE_URL = "https://aeroapi.flightaware.com/aeroapi"

def get_operator_flights(operator_id: str) -> dict:
    url = f"{AEROAPI_BASE_URL}/operators/{operator_id}/flights"
    headers = {"x-apikey": API_KEYS['FA_API_KEY']}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.json()

# Example usage:
data = pd.DataFrame()
for airline in ["UAL", "AAL", "DL", "WN", "AS", "B6"]:
    flights = get_operator_flights(airline)
    flights_df = pd.DataFrame(flights["scheduled"])
    flights_df.to_csv(f"data/{airline}_flights.csv")
    data = pd.concat([data, flights_df]).reset_index(drop=True)
data

/var/folders/mh/b5m2p8fd04x2knz5_hwd1rqw0000gn/T/ipykernel_97836/77999310.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data, flights_df]).reset_index(drop=True)


,ident,ident_icao,ident_iata,actual_runway_off,actual_runway_on,fa_flight_id,operator,operator_icao,operator_iata,flight_number,...,route,baggage_claim,seats_cabin_business,seats_cabin_coach,seats_cabin_first,gate_origin,gate_destination,terminal_origin,terminal_destination,type
0,UAL3938,UAL3938,UA3938,None,None,UAL3938-1778673721-fa-3496p,UAL,UAL,UA,3938,...,CPT DIDZA L9 OKSAW BIBPE BUCGO FELCA NICXI BAK...,None,22.0,NaN,46.0,None,None,None,None,Airline
1,UAL4205,UAL4205,UA4205,None,None,UAL4205-1778519020-fa-2483p,UAL,UAL,UA,4205,...,MLB PBI DOUGZ COREA JOKRS4,None,NaN,NaN,12.0,None,None,None,None,Airline
2,UAL2271,UAL2271,UA2271,None,None,UAL2271-1778449026-fa-0p,UAL,UAL,UA,2271,...,ZIMMR4 CHNGY PIH KU87K PDT CHINS5,None,NaN,NaN,20.0,B29,B11,None,None,Airline
3,UAL122,UAL122,UA122,None,None,UAL122-1778476538-fa-113p,UAL,UAL,UA,122,...,MERIT HFD PUT TUSKY GRAYY NICSO 4800N/05000W 4...,None,21.0,NaN,48.0,C90,None,C,2,Airline
4,UAL1022,UAL1022,UA1022,None,None,UAL1022-1778457862-fa-110p,UAL,UAL,UA,1022,...,RDSOX1 JFRYS PATOY Q116 JAWJA CABLO KT15M KM27...,None,NaN,NaN,24.0,42,B42,B,None,Airline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,ASA1311,ASA1311,AS1311,None,None,ASA1311-1778480241-airline-327p,ASA,ASA,AS,1311,...,ALW EVANT LYNND FMG FRA REBRG IRNMN2,None,NaN,NaN,NaN,C8,66,None,6,Airline
71,ASA73,ASA73,AS73,None,None,ASA73-1778480241-airline-167p,ASA,ASA,AS,73,...,SIT HAWRD5 TNAKY JNU,None,NaN,NaN,NaN,2,2,None,None,Airline
72,ASA5,ASA5,AS5,None,None,ASA5-1778480241-airline-86p,ASA,ASA,AS,5,...,REBLL5 OTTTO Q176 HNN FLM TOY BUM GCK RSK TBC ...,None,NaN,NaN,NaN,B16,66,2,6,Airline
73,ASA33,ASA33,AS33,None,None,ASA33-1778480241-airline-109p,ASA,ASA,AS,33,...,GAYEL Q818 WOZEE KENPA CESNA EXHOS LWT PDT JKN...,None,NaN,NaN,NaN,35,C7,8,None,Airline


## Aircraft Data

In [12]:
from bs4 import BeautifulSoup
import time

_TABLE_TAGS = {"td", "th", "tr", "table", "thead", "tbody", "tfoot"}

def _is_table_selector(css_selector: str) -> bool:
    tokens = css_selector.replace(",", " ").split()
    return bool({t.lower() for t in tokens if t.isalpha()} & _TABLE_TAGS)

def _parse_tables(soup: BeautifulSoup, elements: list) -> pd.DataFrame:
    ancestor_tables = {
        el.find_parent("table") if el.name != "table" else el
        for el in elements
    }
    ancestor_tables.discard(None)

    if not ancestor_tables:
        return pd.DataFrame({"text": [el.get_text(strip=True) for el in elements]})

    dfs = []
    for table in ancestor_tables:
        rows = table.find_all("tr")
        data = [
            [cell.get_text(strip=True) for cell in row.find_all(["td", "th"])]
            for row in rows
        ]
        data = [row for row in data if any(row)]
        if not data:
            continue
        has_header = bool(rows[0].find("th")) if rows else False
        df = pd.DataFrame(data[1:], columns=data[0]) if has_header else pd.DataFrame(data)
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def _fetch_with_requests(url: str) -> str:
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    })
    response = session.get(url, timeout=15)
    response.raise_for_status()
    return response.text

def _fetch_with_selenium(url: str, wait: int = 3) -> str:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    driver = webdriver.Chrome(options=opts)
    try:
        driver.get(url)
        time.sleep(wait)  # allow JS to render
        return driver.page_source
    finally:
        driver.quit()

def scrape_website(url: str, css_selector: str, use_selenium: bool = False, selenium_wait: int = 3) -> "pd.DataFrame | list[str]":
    """
    Scrape a URL using a CSS selector.

    Returns a DataFrame when the selector targets table elements (td, th, tr, table, etc.),
    otherwise returns a list of text strings from matched elements.

    Args:
        url:           Page to scrape.
        css_selector:  CSS selector string, e.g. "td, th" or "h2" or ".price".
        use_selenium:  Use headless Chrome instead of requests (needed for JS-rendered pages).
        selenium_wait: Seconds to wait for JS to render (only used when use_selenium=True).
    """
    html = _fetch_with_selenium(url, wait=selenium_wait) if use_selenium else _fetch_with_requests(url)

    soup = BeautifulSoup(html, "html.parser")
    elements = soup.select(css_selector)

    if not elements:
        print(f"No elements matched selector: '{css_selector}'")
        return pd.DataFrame() if _is_table_selector(css_selector) else []

    if _is_table_selector(css_selector):
        return _parse_tables(soup, elements)

    return [el.get_text(strip=True) for el in elements]
